# MAKE SUBMISSION - ROUND 2 (7 SCENES)
Attach 01_BASE_CODE.zip, 02_ROUND2_DATA.zip, 03_ROUND2_PATCH_V3_DISKSAFE.zip, and the completed model datasets from Group C/D. This notebook automatically selects the highest available iteration for all seven scenes and creates one ZIP below 350 MB.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'CUDA is not available'
print('GPU count:', torch.cuda.device_count())

In [ ]:
from pathlib import Path
import shutil
input_root = Path('/kaggle/input')
work_dir = Path('/kaggle/working/vai_nvs')
if work_dir.exists():
    shutil.rmtree(work_dir)
zip_candidates = list(input_root.rglob('01_BASE_CODE.zip'))
if not zip_candidates:
    zip_candidates = list(input_root.rglob('vai_nvs_kaggle_prod_v3.zip'))
if zip_candidates:
    zip_candidates = sorted(set(zip_candidates), key=lambda p: (len(str(p)), str(p)))
    selected_zip = zip_candidates[0]
    print(f'Found {len(zip_candidates)} Base Code ZIP candidate(s); using: {selected_zip}')
    shutil.unpack_archive(str(selected_zip), str(work_dir))
else:
    candidates = []
    for p in input_root.rglob('gaussian-splatting'):
        parent = p.parent
        is_full_base = (parent / 'scripts' / 'setup_cuda_extensions_linux.sh').exists() and (p / 'submodules' / 'diff-gaussian-rasterization').exists()
        if is_full_base:
            candidates.append(parent)
    candidates = sorted(
        set(candidates),
        key=lambda p: (0 if ('base' in str(p).lower() and 'code' in str(p).lower()) else 1, len(str(p)), str(p)),
    )
    assert candidates, 'BASE CODE missing. Add the Kaggle dataset containing 01_BASE_CODE.zip, then Save & Run All again.'
    selected_root = candidates[0]
    print(f'Found {len(candidates)} extracted Base Code candidate(s); using: {selected_root}')
    if len(candidates) > 1:
        print('Ignoring duplicate source copies from model datasets:', candidates[1:])
    shutil.copytree(selected_root, work_dir)
old_outputs = work_dir / 'outputs'
if old_outputs.exists():
    shutil.rmtree(old_outputs)
assert (work_dir / 'gaussian-splatting').exists()
assert (work_dir / 'scripts' / 'setup_cuda_extensions_linux.sh').exists(), 'Incomplete BASE CODE: setup script missing'
assert (work_dir / 'gaussian-splatting' / 'submodules' / 'diff-gaussian-rasterization').exists(), 'Incomplete BASE CODE: CUDA submodule missing'
print('Base source workspace ready')

In [ ]:
import os
from pathlib import Path
work_dir = Path('/kaggle/working/vai_nvs')
assert work_dir.exists(), 'BASE CODE was not prepared; check the previous cell'
os.chdir(work_dir)
print('Working directory:', Path.cwd())

In [ ]:
!python -m pip install -U pip "setuptools<82" wheel ninja
!pip install tqdm plyfile opencv-python pillow scipy matplotlib

In [ ]:
from pathlib import Path
import shutil
marker_candidates = list(Path('/kaggle/input').rglob('ROUND2_PATCH_V3_DISKSAFE.txt'))
if not marker_candidates:
    hotfix_zips = list(Path('/kaggle/input').rglob('03_ROUND2_PATCH_V3_DISKSAFE.zip'))
    assert len(hotfix_zips) == 1, 'Add exactly one 03_ROUND2_PATCH_V3_DISKSAFE.zip as an input'
    hotfix_extract = Path('/kaggle/working/round2_patch')
    if hotfix_extract.exists():
        shutil.rmtree(hotfix_extract)
    shutil.unpack_archive(str(hotfix_zips[0]), str(hotfix_extract))
    marker_candidates = list(hotfix_extract.rglob('ROUND2_PATCH_V3_DISKSAFE.txt'))
assert len(marker_candidates) == 1, f'Round 2 patch marker problem: {marker_candidates}'
hotfix_root = marker_candidates[0].parent
patches = [
    'gaussian-splatting/scene/cameras.py',
    'gaussian-splatting/render.py',
    'tools/prepare_round2_kaggle.py',
    'tools/undistort_simple_radial_dataset.py',
    'tools/prune_colmap_images_to_existing.py',
    'tools/colmap_io.py',
    'tools/render_test_poses.py',
    'tools/distort_back_renders.py',
    'tools/validate_submission.py',
    'tools/recompress_submission_zip.py',
]
for relative in patches:
    source = hotfix_root / relative
    target = Path('/kaggle/working/vai_nvs') / relative
    assert source.exists(), f'Missing hotfix file: {source}'
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
print('Round 2 patch applied')

In [ ]:
from pathlib import Path
import subprocess
import sys
input_root = Path('/kaggle/input')
round2_scenes = ['HCM0421','HCM0539','HCM0540','HCM0644','HCM0674','bonsai','chair']
candidate_roots = set()
for train_dir in input_root.rglob('HCM0421/train'):
    candidate_roots.add(train_dir.parent.parent)
scored_roots = []
for root in candidate_roots:
    total_images = 0
    complete = True
    for scene in round2_scenes:
        scene_dir = root / scene
        image_dir = scene_dir / 'train' / 'images'
        sparse_dir = scene_dir / 'train' / 'sparse' / '0'
        poses = scene_dir / 'test' / 'test_poses.csv'
        if not image_dir.exists() or not sparse_dir.exists() or not poses.exists():
            complete = False
            break
        total_images += sum(1 for p in image_dir.iterdir() if p.is_file() and not p.name.startswith('._'))
    if complete and total_images > 0:
        scored_roots.append((total_images, root))
scored_roots.sort(key=lambda item: (-item[0], 1 if 'work_round2' in str(item[1]) else 0, len(str(item[1])), str(item[1])))
if scored_roots:
    round2_input = scored_roots[0][1]
    print(f'Using Round 2 raw data root: {round2_input} ({scored_roots[0][0]} training images)')
    if len(scored_roots) > 1:
        print('Ignoring duplicate Round 2 roots:', scored_roots[1:])
else:
    round2_zips = list(input_root.rglob('02_ROUND2_DATA.zip')) or list(input_root.rglob('VAI_NVS_DATA_ROUND2.zip'))
    assert round2_zips, 'Round 2 raw dataset missing: add the dataset containing all seven scenes'
    round2_input = input_root
    print('Using Round 2 ZIP input:', round2_zips[0])
subprocess.run([sys.executable, 'tools/prepare_round2_kaggle.py', '--group', 'ALL', '--input_root', str(round2_input)], check=True)

In [ ]:
from pathlib import Path
import shutil
new_private = Path('work_round2/phase1/private_set1')
assert new_private.exists(), 'Round 2 preprocessing output is missing'
for scene_dir in new_private.iterdir():
    train_dir = scene_dir / 'train'
    if train_dir.exists():
        shutil.rmtree(train_dir)
old_data = Path('work_undistorted')
if old_data.exists():
    shutil.rmtree(old_data)
combined = Path('round2_scenes/phase1/private_set1')
if combined.parent.parent.exists():
    shutil.rmtree(combined.parent.parent)
combined.mkdir(parents=True, exist_ok=True)
for scene_dir in new_private.iterdir():
    if scene_dir.is_dir():
        (combined / scene_dir.name).symlink_to(scene_dir.resolve(), target_is_directory=True)
required = {'HCM0421','HCM0539','HCM0540','HCM0644','HCM0674','bonsai','chair'}
actual = {p.name for p in combined.iterdir() if p.is_dir()}
assert actual == required, f'Round 2 metadata mismatch: missing={required-actual}, extra={actual-required}'
print('Exactly seven Round 2 metadata folders ready')

In [ ]:
from pathlib import Path
import re
import shutil
required = {'HCM0421','HCM0539','HCM0540','HCM0644','HCM0674','bonsai','chair'}
model_candidates = {}
for ply in Path('/kaggle/input').rglob('point_cloud/iteration_*/point_cloud.ply'):
    iteration_match = re.fullmatch(r'iteration_(\d+)', ply.parent.name)
    if not iteration_match:
        continue
    iteration = int(iteration_match.group(1))
    for parent in ply.parents:
        match = re.fullmatch(r'prod_private_([A-Za-z0-9]+)', parent.name)
        if match and match.group(1) in required:
            scene = match.group(1)
            model_candidates.setdefault(scene, []).append((iteration, ply))
            break
missing = required - set(model_candidates)
assert not missing, f'Missing model datasets for: {sorted(missing)}'
model_files = {}
for scene, candidates in model_candidates.items():
    candidates.sort(key=lambda item: (-item[0], len(str(item[1])), str(item[1])))
    model_files[scene] = candidates[0]
    print('SELECTED', scene, 'iteration', candidates[0][0], candidates[0][1])
outputs = Path('outputs')
if outputs.exists():
    shutil.rmtree(outputs)
for scene in sorted(required):
    iteration, ply = model_files[scene]
    target_dir = outputs / f'prod_private_{scene}' / 'point_cloud' / f'iteration_{iteration}'
    target_dir.mkdir(parents=True, exist_ok=True)
    (target_dir / 'point_cloud.ply').symlink_to(ply)
    print(scene, 'iteration', iteration, round(ply.stat().st_size/1024**2,1), 'MiB')
print('All seven Round 2 models linked without copying')

In [ ]:
import os
import torch
major, minor = torch.cuda.get_device_capability(0)
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
print('CUDA architecture:', os.environ['TORCH_CUDA_ARCH_LIST'])
!bash scripts/setup_cuda_extensions_linux.sh

In [ ]:
!python tools/render_test_poses.py --data_root round2_scenes --split private_set1 --model_root outputs --model_template 'prod_{short_split}_{scene}' --output_root outputs/round2_undistorted --output_format png --antialiasing

In [ ]:
!python tools/distort_back_renders.py --data_root round2_scenes --split private_set1 --render_root outputs/round2_undistorted --output_root /kaggle/working/submission_images --layout scene_only --keep_original_extension --copy_if_no_distortion --jpeg_quality 100
from pathlib import Path
import shutil
outputs_root = Path('outputs')
if outputs_root.exists():
    shutil.rmtree(outputs_root)
print('Removed model links and PNG intermediates')

In [ ]:
!python tools/validate_submission.py --data_root round2_scenes --split private_set1 --submission_root /kaggle/working/submission_images --layout scene_only --report_csv /kaggle/working/submission_check.csv --zip_path /kaggle/working/submission_full_quality.zip --max_near_black_fraction 0.05
!python tools/recompress_submission_zip.py /kaggle/working/submission_full_quality.zip /kaggle/working/SUBMISSION_ROUND2.zip --auto_quality --max_bytes 345000000

In [ ]:
from pathlib import Path
import hashlib
import shutil
zip_path = Path('/kaggle/working/SUBMISSION_ROUND2.zip')
assert zip_path.exists() and zip_path.stat().st_size < 350_000_000
sha256 = hashlib.sha256()
with zip_path.open('rb') as stream:
    for chunk in iter(lambda: stream.read(8*1024**2), b''):
        sha256.update(chunk)
print('FINAL ZIP SIZE:', zip_path.stat().st_size, 'bytes')
print('FINAL SHA256:', sha256.hexdigest())
for path in [Path('/kaggle/working/submission_full_quality.zip'), Path('/kaggle/working/submission_images'), Path('/kaggle/working/vai_nvs'), Path('/kaggle/working/round2_patch')]:
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()
print('FINAL ROUND 2 SEVEN-SCENE SUBMISSION READY:', zip_path)